# Chapter 4 Results — Analysis and Visualisation

This notebook loads `results/tables/experiment_results.csv` (produced by `run_experiment.py`)
and reproduces all figures and tables in Chapter 4 of the thesis interactively.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.evaluation.metrics import safe_missingness_threshold

plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})

RESULTS_DIR = Path('../results')
tables_dir  = RESULTS_DIR / 'tables'

df = pd.read_csv(tables_dir / 'experiment_results.csv')
meta = json.loads((tables_dir / 'baseline_meta.json').read_text())
acc_baseline = meta['acc_baseline']

METHOD_LABELS = {
    'Mean_Fill': 'Mean-fill', 'Forward_Fill': 'Forward-fill',
    'Linear': 'Linear', 'Spline': 'Spline',
    'GP_Matern32': 'GP (Matérn-3/2)', 'TS_MICE': 'TS-MICE',
    'KNN_Impute': 'KNN-Impute', 'RF_Impute': 'RF-Impute',
    'RNN_Impute': 'RNN-Impute', 'GAIN_Impute': 'GAIN-Impute',
    'MF_Impute': 'MF-Impute', 'GB_MICE': 'GB-MICE', 'SAITS': 'SAITS',
}
METHOD_ORDER = list(METHOD_LABELS.keys())
FRAC_LABELS = {0.1: '10%', 0.3: '30%', 0.5: '50%'}

print(f'Loaded {len(df)} rows. Baseline accuracy: {acc_baseline:.4f}')
print(df.head())

## 4.1 Reconstruction RMSE and MAE (Table 4.1)

In [ ]:
fractions = sorted(df['fraction'].unique())

for metric in ['rmse', 'mae']:
    pivot = df.pivot_table(values=f'{metric}_mean', index='method', columns='fraction')
    pivot = pivot.reindex([m for m in METHOD_ORDER if m in pivot.index])
    pivot.index = [METHOD_LABELS.get(m, m) for m in pivot.index]
    pivot.columns = [FRAC_LABELS.get(c, str(c)) for c in pivot.columns]
    print(f'\n{metric.upper()} (mean):')
    print(pivot.to_string(float_format='{:.4f}'.format))

## 4.2 Accuracy Loss vs Missingness Fraction

In [ ]:
palette = plt.cm.tab20(np.linspace(0, 1, len(METHOD_ORDER)))
x_vals = [100 * f for f in fractions]

fig, ax = plt.subplots(figsize=(10, 5))
for i, method in enumerate(METHOD_ORDER):
    sub = df[df['method'] == method].sort_values('fraction')
    if sub.empty: continue
    y = sub['acc_loss'].values * 100
    ax.plot(x_vals[:len(y)], y, 'o-', color=palette[i],
            label=METHOD_LABELS.get(method, method), lw=1.5, ms=5)

ax.axhline(0,   color='k',   lw=0.8, ls='--', label='Baseline')
ax.axhline(5,   color='red', lw=0.8, ls=':',  alpha=0.7, label='5 pp threshold')
ax.set_xlabel('Missingness fraction (%)')
ax.set_ylabel('Accuracy loss ΔAcc (pp)')
ax.set_title('Classification accuracy loss vs missingness fraction')
ax.legend(ncol=2, fontsize=7)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 4.3 Classification Table with 95% CIs (Table 4.3)

In [ ]:
for frac in fractions:
    sub = df[df['fraction'] == frac].copy()
    sub = sub.set_index('method').reindex(METHOD_ORDER).reset_index()
    sub['Method'] = sub['method'].map(METHOD_LABELS)
    sub['Accuracy'] = sub['acc_mean'].map('{:.4f}'.format)
    sub['95% CI']   = sub.apply(
        lambda r: f'[{r.acc_ci_lo:.4f}, {r.acc_ci_hi:.4f}]', axis=1)
    sub['ΔAcc (pp)'] = (sub['acc_loss'] * 100).map('{:+.2f}'.format)
    sub['PRR'] = sub['prr'].map('{:.3f}'.format)
    tbl = sub[['Method','Accuracy','95% CI','ΔAcc (pp)','PRR']].dropna(subset=['Method'])
    print(f'\n=== {FRAC_LABELS[frac]} missingness ===')
    print(tbl.to_string(index=False))

## 4.4 Safe Missingness Threshold p* (Table 4.9)

In [ ]:
rows = []
for method in METHOD_ORDER:
    sub = df[df['method'] == method]
    if sub.empty: continue
    acc_loss_map = dict(zip(sub['fraction'], sub['acc_loss']))
    p_star = safe_missingness_threshold(acc_loss_map, threshold=0.05)
    rows.append({'Method': METHOD_LABELS.get(method, method),
                 'p* (5 pp threshold)': f'{p_star:.2f}' if np.isfinite(p_star) else '>0.50'})

pd.DataFrame(rows).set_index('Method')

## 4.5 RMSE Heatmap

In [ ]:
pivot = df.pivot_table(values='rmse_mean', index='method', columns='fraction')
pivot = pivot.reindex([m for m in METHOD_ORDER if m in pivot.index])
pivot.index = [METHOD_LABELS.get(m, m) for m in pivot.index]
pivot.columns = [FRAC_LABELS.get(c, str(c)) for c in pivot.columns]

fig, ax = plt.subplots(figsize=(6, 6))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'RMSE (norm. flux)'})
ax.set_title('Reconstruction RMSE')
ax.set_ylabel('')
fig.tight_layout()
plt.show()